Method 1: Keystroke dynamics

In [1]:
import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from pynput import keyboard

class KeystrokeDynamics:
    def __init__(self):
        self.press_times = {}
        self.release_times = {}
        self.dwell_times = []
        self.flight_times = []
        self.last_key = None
        self.last_release_time = None

    def on_press(self, key):
        try:
            k = key.char
        except AttributeError:
            k = key.name
        self.press_times[k] = time.time()

    def on_release(self, key):
        try:
            k = key.char
        except AttributeError:
            k = key.name

        release_time = time.time()
        self.release_times[k] = release_time

        # Calculate dwell time
        if k in self.press_times:
            dwell_time = release_time - self.press_times[k]
            self.dwell_times.append(dwell_time)

        # Calculate flight time
        if self.last_key is not None and self.last_release_time is not None:
            flight_time = self.press_times[k] - self.last_release_time
            self.flight_times.append(flight_time)

        self.last_key = k
        self.last_release_time = release_time

        if k == 'esc':  # Stop on 'esc'
            return False

    def start_listening(self):
        with keyboard.Listener(on_press=self.on_press, on_release=self.on_release) as listener:
            listener.join()

    def extract_features(self):
        avg_dwell_time = np.mean(self.dwell_times)
        avg_flight_time = np.mean(self.flight_times)
        return [avg_dwell_time, avg_flight_time]

# Example usage with a RandomForest model for classification
kd = KeystrokeDynamics()
kd.start_listening()
features = kd.extract_features()

# Train model (replace with actual training data)
X_train = np.array([[0.2, 0.15], [0.3, 0.18], [0.22, 0.16]])  # Example feature data
y_train = np.array([1, 1, 0])  # Example labels (1 for user, 0 for impostor)
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

# Predict based on extracted features
user_identity = clf.predict([features])
print("User Authenticated:" if user_identity == 1 else "Imposter Detected")


Method 2: Stylometry

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

nltk.download('stopwords')
nltk.download('punkt')

def extract_writing_features(text):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text.lower())
    function_words = [word for word in words if word in stop_words]
    
    function_word_freq = Counter(function_words)
    avg_sentence_length = sum(len(word_tokenize(sent)) for sent in nltk.sent_tokenize(text)) / len(nltk.sent_tokenize(text))

    return function_word_freq, avg_sentence_length

# Example text data
text = "This is an example text. It is used for analyzing the writing style."

# Extract writing style features
function_word_freq, avg_sentence_length = extract_writing_features(text)
print("Function Word Frequencies:", function_word_freq)
print("Average Sentence Length:", avg_sentence_length)

# Example usage of TF-IDF for writing style classification
corpus = ["This is a sample text.", "Another example of user writing.", "More data for writing analysis."]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# Train model (replace with actual training data)
y_train = [1, 1, 0]  # Labels (1 for user, 0 for impostor)
model = SVC(kernel='linear')
model.fit(X, y_train)

# Predict the writing style of a new text
new_text = ["This is a new sample text for testing."]
X_new = vectorizer.transform(new_text)
prediction = model.predict(X_new)
print("User Authenticated:" if prediction == 1 else "Imposter Detected")


Function Word Frequencies: Counter({'is': 2, 'this': 1, 'an': 1, 'it': 1, 'for': 1, 'the': 1})
Average Sentence Length: 7.5
User Authenticated:


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Method 3: Sentiment Analysis

In [3]:
from textblob import TextBlob
from sklearn.ensemble import IsolationForest

def analyze_sentiment(text):
    blob = TextBlob(text)
    sentiment = blob.sentiment.polarity
    return sentiment

# Example text data
text_data = [
    "I love this system, it's amazing!",
    "This project is so frustrating...",
    "I feel neutral about this issue."
]

# Extract sentiment polarity
sentiments = [analyze_sentiment(text) for text in text_data]
print("Sentiment Scores:", sentiments)

# Detect anomalies in sentiment using Isolation Forest
model = IsolationForest(contamination=0.1)
sentiments = [[s] for s in sentiments]  # Convert to 2D array
model.fit(sentiments)

anomalies = model.predict(sentiments)
print("Anomalies Detected (1: normal, -1: anomaly):", anomalies)


Sentiment Scores: [0.625, -0.4, 0.0]
Anomalies Detected (1: normal, -1: anomaly): [-1  1  1]


Method 4: Topic Modelling

In [4]:
from gensim import corpora
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

nltk.download('punkt')
nltk.download('stopwords')

def perform_topic_modeling(texts, num_topics=2):
    stop_words = set(stopwords.words('english'))
    processed_texts = [[word for word in word_tokenize(text.lower()) if word.isalnum() and word not in stop_words] for text in texts]
    
    dictionary = corpora.Dictionary(processed_texts)
    corpus = [dictionary.doc2bow(text) for text in processed_texts]
    
    lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15)
    topics = lda_model.print_topics(num_words=5)
    
    return topics

texts = [
    "Natural language processing is a complex field.",
    "Machine learning is a subset of AI.",
    "Data science involves statistics and programming."
]

# Perform topic modeling
topics = perform_topic_modeling(texts)
for idx, topic in topics:
    print(f"Topic {idx}: {topic}")


Topic 0: 0.094*"statistics" + 0.094*"involves" + 0.094*"science" + 0.094*"programming" + 0.094*"data"
Topic 1: 0.124*"language" + 0.124*"field" + 0.124*"natural" + 0.124*"processing" + 0.124*"complex"


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\conmy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Method 5: Sequence Analysis

In [5]:
from hmmlearn import hmm
import numpy as np

def perform_sequence_analysis(sequences, num_states=2):
    model = hmm.GaussianHMM(n_components=num_states, covariance_type="diag", n_iter=1000)
    model.fit(sequences)
    
    log_likelihood = model.score(sequences)
    return log_likelihood

# Example: Sequence of word lengths
sequences = np.array([[5], [4], [3], [2], [6], [5], [4], [7]])
log_likelihood = perform_sequence_analysis(sequences)
print("Log Likelihood:", log_likelihood)


Log Likelihood: -14.006010969562102


COMBINATION

In [7]:
import numpy as np

class ContinuousAuthenticator:
    def __init__(self):
        self.keystroke_model = RandomForestClassifier()  # Trained elsewhere
        self.stylometry_model = SVC(kernel='linear')  # Trained elsewhere
        self.sentiment_model = IsolationForest(contamination=0.1)
        self.hmm_model = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=1000)
        
    def authenticate(self, keystroke_features, writing_features, sentiments, sequence):
        # Keystroke Dynamics
        kd_score = self.keystroke_model.predict([keystroke_features])[0]
        
        # Stylometry
        stylometry_score = self.stylometry_model.predict([writing_features])[0]
        
        # Sentiment Analysis
        sentiment_scores = [[s] for s in sentiments]
        sentiment_score = np.mean(self.sentiment_model.predict(sentiment_scores))
        
        # Sequence Analysis
        seq_score = self.hmm_model.score(sequence)
        
        # Combine scores (example weights, adjust as needed)
        final_score = 0.4 * kd_score + 0.3 * stylometry_score + 0.2 * sentiment_score + 0.1 * seq_score
        
        return final_score >= threshold  # threshold is set based on training
        
# Usage
authenticator = ContinuousAuthenticator()
keystroke_features = [0.2, 0.15]
writing_features = [0.12, 15.3]
sentiments = [0.5, -0.3, 0.1]
sequence = np.array([[5], [4], [3], [2], [6]])
is_authenticated = authenticator.authenticate(keystroke_features, writing_features, sentiments, sequence)
print("User Authenticated:" if is_authenticated else "Imposter Detected")


NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.